[CAPSTONE INTEGRATION TARGET]
    Build the constrained rectangular stock that becomes a projector mounting bracket. A stable profile is the foundation for hardware dimensions that can change without moving the design origin.


# 1. A rectangle with intent
    Learn to distinguish raw coordinates from driving constraints, explain sixteen initial degrees of freedom, and predict a pad volume before executing CAD. Work offline first. The generated source targets FreeCAD Python 3.11+; this notebook uses Python 3.12+.

    The Build environment team owns the phase branch. DevOps owns connectivity and live acceptance. Set `CORTEXCAD_LIVE=1` only when those prerequisites are met; running the default notebook never opens a connection.


In [ ]:
from __future__ import annotations
import importlib.util
import json
import math
import os
from pathlib import Path
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Choose the CortexCAD Python 3.12+ Jupyter kernel")

# Installed packages work from any directory. A source checkout can be selected
# explicitly; ancestor discovery also supports launching Jupyter inside a phase.
if importlib.util.find_spec("cortexcad") is None:
    root_hint: str = os.environ.get("CORTEXCAD_ROOT", "")
    candidates: list[Path] = ([Path(root_hint).expanduser()] if root_hint else [])
    candidates += [Path.cwd(), *Path.cwd().parents]
    source_root: Path | None = next((path for path in candidates
                                    if (path / "cortexcad" / "recipes.py").is_file()), None)
    if source_root is None:
        raise RuntimeError("Install CortexCAD in this kernel or set CORTEXCAD_ROOT to its checkout")
    sys.path.insert(0, str(source_root.resolve()))

from cortexcad.client import FreeCADClient
from cortexcad.recipes import DEFAULTS, build_script, validate_parameters
LIVE: bool = os.environ.get("CORTEXCAD_LIVE") == "1"
print("LIVE execution enabled" if LIVE else "OFFLINE: no CAD connection will be opened")


In [ ]:
PHASE: str = "sketcher"
params: dict[str, float] = validate_parameters(PHASE, {"length": 60, "width": 40, "height": 6})
expected_volume: float = params["length"] * params["width"] * params["height"]
assert math.isclose(expected_volume, 14400.0)
print(params, "Expected pad volume (mm³):", expected_volume)


## 2. Predict the constraint system
    Four segments have 16 scalar degrees of freedom. Four coincident endpoint pairs remove 8; horizontal/vertical constraints remove 4; fixing the first point at the origin removes 2; length and width remove 2. This count assumes independence. The live recipe requires `solve() == 0` and `FullyConstrained == True` after recompute.

    Find each of those constraints in the generated source below. Notice that a Pad is created inside a PartDesign Body. The sketch is not a decorative rectangle: it supplies the pad profile.


In [ ]:
script: str = build_script(PHASE, params)
compile(script, "freecad_lesson.py", "exec")
print(script)
print("Compiled source only: no FreeCAD solver or solid has run on this host.")


## 3. Submit once, then inspect
    The live recipe creates `CortexCAD_Sketcher`, requires one valid solid and compares its volume with the analytic prediction. Existing documents are never silently replaced. Save and close an earlier lesson document explicitly before a deliberate rebuild. After any timeout, inspect the workstation before another request.


In [ ]:
if LIVE:
    # Endpoint and explicit timeout come from CORTEXCAD_ENDPOINT/TIMEOUT.
    client: FreeCADClient = FreeCADClient(timeout=float(os.environ.get("CORTEXCAD_TIMEOUT", "5")))
    outcome = client.run_recipe(PHASE, params)
    print(repr(outcome))
    print("Inspect the workstation and native assertions before accepting this lesson.")
else:
    print("OFFLINE preview complete; native geometry and hardware-fit gates remain pending.")


## 4. Falsify invalid inputs
    The following checks run locally in both modes. They must reject bad dimensions without requiring FreeCAD.


In [ ]:
for invalid in ({"height": 0}, {"length": True}, {"length": float("nan")}, {"unknown": 1}):
    try:
        build_script(PHASE, invalid)
    except ValueError as error:
        print(type(error).__name__, str(error))
    else:
        raise AssertionError(f"Unsafe parameter was accepted: {invalid}")


## 5. Transfer the learning
    Change width to 52 mm and predict 18,720 mm³. In a disposable saved FreeCAD copy, remove the width constraint, recompute, and show that the sketch is no longer fully constrained. Restore it before acceptance.

    Record actual solver status, actual solid count, actual volume, FreeCAD/Python versions, and the saved document. No output from an offline run is native geometry evidence. Follow the phase README's console checks, then continue to the Boolean lab.
